# สร้าง AI Agent ขายกาแฟ

เมื่อวานเรากดปุ่มเอง วันนี้ให้โปรแกรมทำแทน ทีละขั้น:

1. **agent ด้วยกฎ** — วนดูตลาด ตัดสินใจด้วย Python ส่งข้อเสนอเอง
2. **เรียก LLM ครั้งแรก** — ขอคำตอบเป็น JSON และรับมือเมื่อ LLM พลาด
3. **ให้ LLM ช่วยตัดสินใจ** — LLM เลือกตัวคูณราคา Python คุมตัวเลข
4. **starter agent** — agent ตั้งต้นที่ทีมจะเอาไปแก้ต่อจนถึงวันแข่ง

**วิธีใช้:** รันทีละช่อง (Shift+Enter) จากบนลงล่าง · ช่องที่วนเล่นตลาดหยุดได้ด้วยปุ่ม ■ (Interrupt)

ถ้าขึ้น `No module named ...` ให้รันใน Terminal: `pip install httpx openai`

## 0. ตั้งค่า

ใช้ **ตลาดฝึก** (sandbox) และ key ของทีม · LLM ใช้ gateway ของผู้จัด (key เดียวกับตลาด) · **ใส่ key ที่บรรทัด `KEY` ที่เดียว** `LLM_KEY` ไม่ต้องแก้
ฝึกนอกเวลาแข่งจะใช้ Ollama บนเครื่องตัวเองก็ได้: `LLM_URL = "http://localhost:11434/v1"`, `LLM_MODEL = "qwen3.5:4b"` — **ช่วงแข่ง 2–3 ใช้ได้เฉพาะ gateway ของผู้จัด** LLM อื่น (cloud หรือ Ollama) ผิดกติกา ตัดสิทธิ์

In [ ]:
import json
import time

import httpx
from openai import OpenAI

MARKET = "http://<ที่อยู่ตลาดฝึก>:8000"      # ทีมงานจะแจ้งในห้อง
KEY = "sk-T00-ใส่key-ของทีม"                 # key ของทีม ใส่ที่นี่ที่เดียว (ขึ้นต้นด้วย sk-T)

LLM_URL = "http://<ที่อยู่-LLM-gateway>:4000/v1"
LLM_KEY = KEY                                # ไม่ต้องแก้: gateway ใช้ key เดียวกับตลาด
LLM_MODEL = "event-llm"

# ตรวจก่อนว่าแก้ที่อยู่และ key แล้ว (ถ้าไม่ตรวจ จะได้ error ภาษาอังกฤษที่อ่านยาก)
if "<" in MARKET or "<" in LLM_URL:
    raise ValueError("ยังไม่ได้ใส่ที่อยู่: แก้ MARKET และ LLM_URL ในช่องนี้ตามที่ทีมงานแจ้ง")
if not KEY.isascii() or not KEY.startswith("sk-T") or KEY.startswith("sk-T00-"):
    raise ValueError("ยังไม่ได้ใส่ key ของทีม: แก้ KEY ในช่องนี้เป็น key ที่ทีมงานแจก (ขึ้นต้นด้วย sk-T ไม่มีภาษาไทย)")

api = httpx.Client(base_url=MARKET, headers={"Authorization": f"Bearer {KEY}"}, timeout=10)
llm = OpenAI(base_url=LLM_URL, api_key=LLM_KEY, timeout=60, max_retries=0)
print(api.get("/v1/version").json())

## 1. agent ด้วยกฎ

### 1.1 เครื่องมือจากเมื่อวาน: สูตร TV และการเช็กสเปก

วันนี้ผลวัดตัวอย่าง **คลาดเคลื่อนได้** — TV ที่คำนวณเป็นแค่ค่าประมาณ และล็อตที่ตัวอย่างผ่านสเปกพอดี ของจริงอาจไม่ผ่าน (`SPEC_MISMATCH`)

In [ ]:
# /v1/calendar ตอบสองส่วน: calendar (ปฏิทินตลาด) และ rules (ตัวเลขของสูตรมูลค่า กระสอบ คะแนน เวลา)
# ใช้แค่ rules · ดูคำตอบเต็มและความหมายทุกช่องได้ใน notebook เมื่อวาน (เรียกตลาดด้วย-Python ข้อ 4)
rules = api.get("/v1/calendar").json()["rules"]
BAG = rules["bag_kg"]                          # 1 กระสอบ = 60 กก.

def grade(d):
    """เกรดและตัวคูณขั้นเกรด จากข้อบกพร่อง d %"""
    for step in rules["defect_steps"]:
        if d <= step["max_defect_pct"]:
            return step["grade"], step["step"]

def true_value(sample):
    """TV = P0 × G × M × S จากผลวัดตัวอย่าง (วันนี้เป็นแค่ค่าประมาณ)"""
    d, m, s = sample["defect_pct"], sample["moisture_pct"], sample["screen"]
    mo = rules["moisture"]                    # M: ตัวคูณความชื้น
    if m <= mo["ref"]:
        mf = 1.0
    elif m <= mo["hard_limit"]:
        mf = (1 - m / 100) / (1 - mo["ref"] / 100) - mo["penalty_per_pp"] * (m - mo["ref"])
    else:
        mf = mo["hard_factor"]
    screens = {int(k): v for k, v in rules["screen_factors"].items()}
    sf = screens[min(max(s, min(screens)), max(screens))]
    return rules["p0"] * (1 - d / 100) * grade(d)[1] * mf * sf

def passes(sample, spec):
    """ผลวัดตัวอย่างผ่านสเปกของผู้ซื้อทุกข้อไหม"""
    return (sample["defect_pct"] <= spec["max_defect_pct"]
            and sample["moisture_pct"] <= spec["max_moisture_pct"]
            and (spec["min_screen"] is None or sample["screen"] >= spec["min_screen"]))

state = api.get("/v1/state").json()
for lot in state["lots"]:
    print(lot["lot_id"], lot["sample"], grade(lot["sample"]["defect_pct"])[0], round(true_value(lot["sample"]), 2))

### 1.2 เดินเกมทีละขั้น: ทำทีละช่องก่อน แล้วค่อยให้ลูปทำแทน

เกมหนึ่งเกมในตลาดฝึกเดินตาม **ช่วง (phase)** เหล่านี้ ตลาดเปลี่ยนช่วงเองตามเวลา agent ต้องดูว่าตอนนี้อยู่ช่วงไหน แล้วทำให้ตรงช่วง

| ช่วง | นานราว (ตลาดฝึกวันนี้) | ในตลาดเกิดอะไร | agent ทำอะไร |
|---|---|---|---|
| `SETUP` | 5 วินาที | แจกล็อตให้ทีม | รอ |
| `APPRAISAL` | 1 นาที | ยังไม่มีผู้ซื้อ (RFQ) | **ประเมินราคา:** ส่ง TV ทุกล็อต ส่งได้ครั้งเดียวต่อเกม |
| `ROUND_OPEN` | 90 วินาทีต่อรอบ | RFQ ของรอบขึ้น | **ขาย:** เลือก RFQ ล็อต กระสอบ ราคา แล้วส่ง |
| `ROUND_CLEARING` | 8 วินาที | ตลาดจับคู่ ประกาศผลรอบ | **อ่านผล** เก็บไว้ใช้รอบหน้า |
| `CLOSED` | หลังรอบที่ 8 | จบเกม คิดคะแนน | หยุด |

`ROUND_OPEN` → `ROUND_CLEARING` วนซ้ำ 8 รอบ · เวลาข้างบนเป็นของตลาดฝึกวันนี้ วันแข่งยาวกว่า ให้อ่านเวลาที่เหลือจาก `seconds_left` เสมอ

**ทำ 6 ขั้นข้างล่างตามลำดับ** (1 เกม 1 รอบขาย):
1. เริ่มเกมใหม่ → 2. ประเมินราคา → 3. รอรอบเปิด → 4. ตัดสินใจ → 5. ส่งข้อเสนอ → 6. อ่านผล

ช่วงประเมินสั้น: รันขั้นที่ 1 แล้วรันขั้นที่ 2 ต่อทันที

In [ ]:
def phase_line(s):
    """บอกสั้นๆ ว่าตลาดอยู่ช่วงไหน รอบที่เท่าไร เหลือกี่วินาที"""
    rnd = f" รอบ {s['round']['number']}/{s['round']['of']}" if s.get("round") else ""
    left = "" if s["seconds_left"] is None else f" · เหลือ {s['seconds_left']:.0f} วินาที"
    return f"ช่วง {s['phase']}{rnd}{left}"

def wait_for(phase):
    """รอจนตลาดเข้าช่วง phase (หรือปิดตลาด) แล้วคืนสถานะล่าสุด · หยุดรอได้ด้วยปุ่ม ■"""
    while True:
        s = api.get("/v1/state").json()
        print("\r" + phase_line(s) + " " * 10, end="")
        if s["phase"] in (phase, "CLOSED"):
            print()
            return s
        time.sleep(1)

print(phase_line(api.get("/v1/state").json()))

**ขั้นที่ 1 · เริ่มเกมใหม่** (เหมือนข้อ 3 เมื่อวาน)

เกมเดิมของทีมจบทันที ได้ล็อตชุดใหม่ แล้วตลาดเดินเอง: `SETUP` → `APPRAISAL` → รอบ 1 …

In [ ]:
r = api.post("/v1/sandbox/reset")      # เกมเดิมของทีมจบทันที ได้ล็อตชุดใหม่
s = r.json()
print(r.status_code, phase_line(s), "· ได้", len(s["lots"]), "ล็อต")

**ขั้นที่ 2 · ประเมินราคา (ช่วง `APPRAISAL`)**

ส่ง TV ที่คำนวณด้วยสูตรในข้อ 1.1 ของทุกล็อตเป็น **ผลประเมิน** (`POST /v1/appraisals`)
- ส่งได้ **ครั้งเดียวต่อเกม** และส่งได้เฉพาะช่วงนี้ · ช่วงนี้ยังไม่มี RFQ จึงยังขายไม่ได้
- คะแนนประเมินคิดจากความใกล้ TV จริง · ไม่ส่ง = คะแนนประเมิน 0
- TV จริงดูได้ตอนจบเกม (ข้อ 4.4)

In [ ]:
s = api.get("/v1/state").json()
if s["phase"] == "SETUP":
    s = wait_for("APPRAISAL")                  # ตลาดกำลังแจกล็อต รออีกไม่กี่วินาที

if s["appraisal_submitted"]:
    print("เกมนี้ส่งผลประเมินไปแล้ว (ส่งได้ครั้งเดียว) · ไปขั้นที่ 3")
elif s["phase"] != "APPRAISAL":
    print(phase_line(s), "· เลยช่วงประเมินแล้ว: รันขั้นที่ 1 ใหม่ แล้วรันช่องนี้ต่อทันที")
else:
    items = [{"lot_id": l["lot_id"], "est_value_per_kg": round(true_value(l["sample"]), 2)} for l in s["lots"]]
    for it in items:
        print(it["lot_id"], "ประเมิน", it["est_value_per_kg"], "บาท/กก.")
    r = api.post("/v1/appraisals", json={"items": items})
    print("ส่งผลประเมิน", r.status_code, "· สำเร็จ" if r.is_success else r.json()["error"])

**ขั้นที่ 3 · รอรอบเปิด (ช่วง `ROUND_OPEN`)**

RFQ ขึ้นเฉพาะตอนรอบเปิด ช่องนี้รอให้เอง แล้วแสดง RFQ ของรอบ

ใหม่วันนี้: **จำกัดกระสอบต่อรอบ** — ทีมเสนอรวมได้ไม่เกิน `capacity_left_kg` ต่อรอบ จึงเทขายหมดในรอบเดียวไม่ได้

In [ ]:
s = wait_for("ROUND_OPEN")
print("RFQ รอบนี้", len(s["rfqs"]), "รายการ · รอบนี้ทีมเสนอรวมได้อีก", s["capacity_left_kg"], "กก.")
for q in s["rfqs"]:
    print(q["rfq_id"], q["buyer_type"], q["buyer"], q["bags"], "กระสอบ", q["spec"])

**ขั้นที่ 4 · สมองของ agent: ตัดสินใจหนึ่งรอบ**

กฎง่ายๆ: ผู้ซื้อที่จ่ายดีได้ของก่อน · ล็อตมูลค่าสูงก่อน · ราคา = ตัวคูณของประเภทผู้ซื้อ × TV

ช่องนี้แค่ **คิด** ยังไม่ส่ง · รอบเปิด 90 วินาที ถ้าขั้นที่ 5 ขึ้นว่ารอบปิดแล้ว ให้กลับไปขั้นที่ 3 รอรอบถัดไป

In [ ]:
MARKUP = {"SPECIALTY": 1.00, "COMMERCIAL": 1.00, "INSTANT": 1.00}   # ลองแก้แล้วเทียบผล

def decide_rules(state, markup):
    tv = {l["lot_id"]: true_value(l["sample"]) for l in state["lots"]}
    left = {l["lot_id"]: l["kg_available"] for l in state["lots"]}
    room = state["capacity_left_kg"] if state["capacity_left_kg"] is not None else 10**9
    offers = []
    for q in sorted(state["rfqs"], key=lambda q: -markup[q["buyer_type"]]):
        for lot in sorted(state["lots"], key=lambda l: -tv[l["lot_id"]]):
            if not passes(lot["sample"], q["spec"]):
                continue
            kg = min(left[lot["lot_id"]], q["kg"], room) // BAG * BAG
            if kg < BAG:
                continue
            offers.append({"rfq_id": q["rfq_id"], "lot_id": lot["lot_id"], "kg": kg,
                           "price_per_kg": round(markup[q["buyer_type"]] * tv[lot["lot_id"]], 2)})
            left[lot["lot_id"]] -= kg
            room -= kg
            break                     # RFQ ละ 1 ข้อเสนอ
    return offers


offers = decide_rules(s, MARKUP)
for o in offers:
    print(o)
print(len(offers), "ข้อเสนอ ·", sum(o["kg"] for o in offers) // BAG, "กระสอบ")

**ขั้นที่ 5 · ส่งข้อเสนอ**

`POST /v1/offers:batch` ส่งทุกข้อเสนอในครั้งเดียว · `replace_existing: true` = ยกเลิกของเดิมในรอบนี้ก่อน (ส่งซ้ำได้ไม่ซ้อน)
ถ้ามีข้อเสนอผิดแม้ตัวเดียว ทั้งชุดถูกปฏิเสธ (`BATCH_REJECTED`) พร้อมบอกว่าตัวไหนผิด

In [ ]:
if not offers:
    print("ไม่มีข้อเสนอให้ส่ง: ไม่มีล็อตไหนผ่านสเปก RFQ รอบนี้ · รอรอบถัดไป (ขั้นที่ 3)")
else:
    r = api.post("/v1/offers:batch", json={"offers": offers, "replace_existing": True})
    print(r.status_code, "ส่ง", len(offers), "ข้อเสนอ")
    if r.status_code >= 400:
        print(r.json()["error"])               # WRONG_PHASE = รอบปิดแล้ว: กลับไปขั้นที่ 3

**ขั้นที่ 6 · อ่านผลรอบ (ช่วง `ROUND_CLEARING`)**

พอรอบปิด ตลาดจับคู่ข้อเสนอกับผู้ซื้อ แล้วใส่ผลไว้ใน `last_results` ของสถานะ

| สถานะ | ความหมาย | ทำอย่างไรต่อ |
|---|---|---|
| `ACCEPTED` | ขายได้ | ขายได้ตลอด = อาจถูกไป ลองขึ้นราคา |
| `PARTIAL` | ขายได้บางส่วน ผู้ซื้อซื้อครบก่อน | ส่วนที่เหลือยังขายรอบหน้าได้ |
| `ABOVE_LIMIT` | แพงเกินเพดานของผู้ซื้อ | ลดตัวคูณของประเภทนั้น |
| `OUTBID` | ทีมอื่นเสนอคุ้มกว่า และผู้ซื้อซื้อครบแล้ว | ลดราคา หรือไป RFQ อื่น |
| `SPEC_MISMATCH` | ของจริงไม่ผ่านสเปก (ตัวอย่างคลาดเคลื่อน) | เลี่ยงล็อตที่ผ่านสเปกแบบเฉียดๆ |

In [ ]:
rid = s["round"]["round_id"]
print("รอผลรอบ", rid)
while True:
    st = api.get("/v1/state").json()
    res = st.get("last_results") or {}
    if res.get("round_id") == rid or st["phase"] == "CLOSED":
        break
    print("\r" + phase_line(st) + " " * 10, end="")
    time.sleep(1)
print()
for o in res.get("my_offers", []):
    print(o["rfq_id"], o["lot_id"], o["price_per_kg"], "บาท/กก.", o["status"],
          f"ขายได้ {o['filled_kg'] // BAG}/{o['kg'] // BAG} กระสอบ")

ครบ 1 รอบแล้ว: **ดู → ตัดสินใจ → ส่ง → อ่านผล** · แต่เกมมี 8 รอบ และตลาดไม่รอเรา ทำมือทุกรอบไม่ทัน → ข้อ 1.3 ให้ลูปทำแทน

### 1.3 ลูปของ agent: ทำขั้นที่ 2–6 ซ้ำเอง

ทุก 1 วินาที ลูปอ่าน `GET /v1/state` แล้วทำตามช่วงของตลาด เหมือนที่เราทำมือในข้อ 1.2

| เห็นอะไร | ลูปทำอะไร | ขั้นในข้อ 1.2 |
|---|---|---|
| ช่วง `APPRAISAL` และยังไม่ได้ส่ง | ส่งผลประเมิน | ขั้นที่ 2 |
| ช่วง `ROUND_OPEN` รอบที่ยังไม่ได้ทำ | ตัดสินใจ แล้วส่งข้อเสนอ ครั้งเดียวต่อรอบ | ขั้นที่ 4–5 |
| ผลรอบใหม่ใน `last_results` | พิมพ์ผล และจำไว้ใน `history` (ใช้ในข้อ 3) | ขั้นที่ 6 |
| ช่วง `CLOSED` | หยุด | |

- เน็ตสะดุดก็ไม่หยุด: รอแล้วลองใหม่
- ถ้าเริ่มรันตอนเลยช่วงประเมินไปแล้ว ลูปจะเตือน

**ช่องถัดไป: ตัวช่วยแสดงผล และ `history`** (ยังไม่มีอะไรรัน แค่สร้างฟังก์ชันไว้ให้ `run()` เรียก)

| ชื่อ | ทำอะไร |
|---|---|
| `history` | รายการ (list) ผลของเรา **1 บรรทัดต่อรอบ** เป็นภาษาอังกฤษสั้นๆ เช่น `R1: COMMERCIAL x1.00 ACCEPTED 4/4 bags` · ข้อ 1 ยังไม่ได้ใช้ · **ข้อ 3 ส่ง 5 บรรทัดล่าสุดให้ LLM อ่าน** เพื่อให้ LLM รู้ผลรอบก่อนๆ (LLM จำเองไม่ได้) |
| `show_offers()` | พิมพ์ข้อเสนอที่เพิ่งส่ง: RFQ ไหน ผู้ซื้อประเภทอะไร ล็อตไหน กี่กระสอบ ราคาเท่าไร ตัวคูณเท่าไร |
| `show_results()` | พิมพ์ผลรอบทีละข้อเสนอ: สถานะ ขายได้กี่กระสอบ ได้เงินเท่าไร แล้วต่อท้าย `history` 1 บรรทัด · คืนรายได้ของรอบ |
| `show_stock()` | พิมพ์กระสอบที่เหลือของแต่ละล็อต (เหลือ/ทั้งหมด) |

In [ ]:
history = []          # ผลของเรา 1 บรรทัดต่อรอบ (ภาษาอังกฤษ) · ข้อ 3 ส่งให้ LLM อ่าน

def show_offers(offers, s, tv):
    """พิมพ์ข้อเสนอที่เพิ่งส่ง"""
    types = {q["rfq_id"]: q["buyer_type"] for q in s["rfqs"]}
    for o in offers:
        print(f"   {o['rfq_id']} {types.get(o['rfq_id'], '?'):<10} ← {o['lot_id']}  {o['kg'] // BAG} กระสอบ × "
              f"{o['price_per_kg']:.2f} บาท/กก. (x{o['price_per_kg'] / tv[o['lot_id']]:.2f} ของ TV)")

def show_results(res, tv):
    """พิมพ์ผลรอบทีละข้อเสนอ แล้วเก็บสรุป 1 บรรทัดไว้ใน history · คืนรายได้ของรอบ (บาท)"""
    types = {r["rfq_id"]: r["buyer_type"] for r in res["market_summary"]}
    print(f"   ผลรอบ {res['round_id']}:")
    parts, income, bags = [], 0, 0
    for o in res["my_offers"]:
        kind = types.get(o["rfq_id"], "?")
        mk = o["price_per_kg"] / tv[o["lot_id"]] if o["lot_id"] in tv else 0
        thb = o["filled_kg"] * o["price_per_kg"]          # ได้เงิน = กก. ที่ขายได้ × ราคาที่เราเสนอ
        income += thb
        bags += o["filled_kg"] // BAG
        print(f"     {o['rfq_id']} {kind:<10} ← {o['lot_id']}  x{mk:.2f}  {o['status']:<13} "
              f"ขายได้ {o['filled_kg'] // BAG}/{o['kg'] // BAG} กระสอบ" + (f" = {thb:,.0f} บาท" if thb else ""))
        parts.append(f"{kind} x{mk:.2f} {o['status']} {o['filled_kg'] // BAG}/{o['kg'] // BAG} bags")
    print(f"   รอบนี้ขายได้ {bags} กระสอบ · {income:,.0f} บาท")
    history.append(f"{res['round_id']}: " + ("; ".join(parts) or "no offers"))
    return income

def show_stock(s):
    """พิมพ์กระสอบที่เหลือของแต่ละล็อต"""
    left = sum(l["kg_remaining"] for l in s["lots"]) // BAG
    total = sum(l["kg_total"] for l in s["lots"]) // BAG
    print("   สต็อก (เหลือ/ทั้งหมด): " + " · ".join(f"{l['lot_id']} {l['kg_remaining'] // BAG}/{l['kg_total'] // BAG}"
                                               for l in s["lots"]) + f" · รวมเหลือ {left}/{total} กระสอบ")

**ช่องถัดไป: `run(decide, max_rounds)` ลูปของ agent** (สร้างฟังก์ชันไว้ ยังไม่เริ่มลูป)

- `decide` = **สมองของ agent** เป็นฟังก์ชันที่รับสถานะตลาด `s` แล้วคืนรายการข้อเสนอ · ลูปไม่สนว่าสมองคิดอย่างไร จึงเปลี่ยนสมองได้โดยไม่แก้ลูป: ข้อ 1 ใช้กฎ (`decide_rules`) ข้อ 3 ใช้ LLM (`decide_llm`)
- `max_rounds` = ขายกี่รอบแล้วหยุด · ไม่ใส่ = เล่นจนตลาดปิด

ในลูป ทุก 1 วินาทีทำ 5 อย่างตามลำดับ (ดูเลข ①–⑤ ในโค้ด):

| | ทำอะไร | ขั้นในข้อ 1.2 |
|---|---|---|
| ① | อ่านสถานะตลาด · เน็ตสะดุดก็รอ 3 วินาทีแล้วลองใหม่ ไม่หยุด | |
| ② | ช่วง `APPRAISAL` และยังไม่ได้ส่ง → ส่งผลประเมิน | ขั้นที่ 2 |
| ③ | มีผลรอบใหม่ → พิมพ์ผล สต็อก รายได้รวม และต่อท้าย `history` | ขั้นที่ 6 |
| ④ | รอบเปิดและยังไม่ได้ทำรอบนี้ → เรียก `decide(s)` แล้วส่งข้อเสนอ | ขั้นที่ 4–5 |
| ⑤ | ตลาดปิด → หยุด | |

ลูปวนทุกวินาที จึงต้องจำว่าทำอะไรไปแล้ว: `acted` = รอบที่ส่งข้อเสนอแล้ว (ไม่ส่งซ้ำ) · `seen` = รอบที่พิมพ์ผลแล้ว (ไม่พิมพ์ซ้ำ)

In [ ]:
def run(decide, max_rounds=None):
    """ลูปของ agent: ดูตลาดทุก 1 วินาที แล้วทำตามช่วง จนตลาดปิดหรือครบ max_rounds รอบ"""
    acted = set()         # รอบที่ส่งข้อเสนอแล้ว
    seen = set()          # รอบที่พิมพ์ผลแล้ว
    income = 0            # รายได้รวมตั้งแต่เริ่มลูป (บาท)
    first = True
    while True:
        # ① ดู: อ่านสถานะตลาด
        try:
            s = api.get("/v1/state").json()
        except httpx.HTTPError as e:              # เน็ตสะดุด: รอแล้วลองใหม่ อย่าให้ agent ตาย
            print("อ่านสถานะไม่ได้:", e)
            time.sleep(3)
            continue
        tv = {l["lot_id"]: true_value(l["sample"]) for l in s["lots"]}
        if first and s["phase"] in ("ROUND_OPEN", "ROUND_CLEARING", "CLOSED") and not s["appraisal_submitted"]:
            print("เกมนี้เลยช่วงประเมินแล้ว และยังไม่ได้ส่ง: คะแนนประเมินเป็น 0 · เริ่มเกมใหม่ด้วยช่องเริ่มเกมใหม่ข้างล่าง")
        first = False

        # ② ช่วงประเมิน: ส่งผลประเมินครั้งเดียวต่อเกม
        if s["phase"] == "APPRAISAL" and not s["appraisal_submitted"]:
            items = [{"lot_id": i, "est_value_per_kg": round(v, 2)} for i, v in tv.items()]
            r = api.post("/v1/appraisals", json={"items": items})
            print("ส่งผลประเมิน", len(items), "ล็อต ·", r.status_code)

        # ③ มีผลรอบใหม่: พิมพ์ผล สต็อก และรายได้รวม
        res = s.get("last_results")
        if res and res.get("round_id") and res["round_id"] not in seen:
            seen.add(res["round_id"])
            income += show_results(res, tv)
            show_stock(s)
            print(f"   ขายแล้วรวม {income:,.0f} บาท")

        # ④ รอบเปิดใหม่: ตัดสินใจ แล้วส่งข้อเสนอ
        if s["phase"] == "ROUND_OPEN" and s["round"]["round_id"] not in acted:
            if max_rounds and len(acted) >= max_rounds:
                print(f"\nครบ {max_rounds} รอบแล้ว หยุด · ขายแล้วรวม {income:,.0f} บาท")
                return
            acted.add(s["round"]["round_id"])
            rnd, cap = s["round"], s["capacity_left_kg"]
            print(f"\n━━ รอบ {rnd['round_id']} ({rnd['number']}/{rnd['of']}) · RFQ {len(s['rfqs'])} รายการ · "
                  f"เสนอได้ {'ไม่จำกัด' if cap is None else cap // BAG} กระสอบ ━━")
            t0 = time.time()
            offers = decide(s)                    # สมองของ agent คิดตรงนี้
            r = api.post("/v1/offers:batch", json={"offers": offers, "replace_existing": True}) if offers else None
            print(f"   ส่ง {len(offers)} ข้อเสนอ {sum(o['kg'] for o in offers) // BAG} กระสอบ · "
                  f"คิด {time.time() - t0:.1f} วิ · " + (str(r.status_code) if r else "ไม่มีข้อเสนอ"))
            show_offers(offers, s, tv)
            if r is not None and r.status_code >= 400:
                print("   ", r.json()["error"])

        # ⑤ ตลาดปิด: หยุด
        if s["phase"] == "CLOSED":
            print(f"\nตลาดปิด · ขายแล้วรวม {income:,.0f} บาท · ดูคะแนนและ TV จริงในข้อ 4.4")
            return
        time.sleep(1)

**เริ่มเล่น: รันสองช่องข้างล่างตามลำดับ** เริ่มเกมใหม่ → ลูป · หยุดลูปได้ด้วยปุ่ม ■

บรรทัด `run(lambda s: decide_rules(s, MARKUP), max_rounds=3)` อ่านว่า "เริ่มลูป ใช้กฎในข้อ 1.2 เป็นสมอง ขาย 3 รอบแล้วหยุด"
- `run()` ส่งให้สมองแค่สถานะ `s` อย่างเดียว แต่ `decide_rules` ต้องการ 2 อย่าง (สถานะ และตัวคูณ `MARKUP`)
- `lambda s: decide_rules(s, MARKUP)` คือฟังก์ชันสั้นๆ ไม่มีชื่อ รับ `s` แล้วเรียก `decide_rules(s, MARKUP)` · เขียนยาวได้แบบนี้ ผลเหมือนกัน:
  ```python
  def brain(s):
      return decide_rules(s, MARKUP)
  run(brain, max_rounds=3)
  ```
- `max_rounds=3` = ขาย 3 รอบแล้วหยุด (เกมเต็มมี 8 รอบ) · ลบออก = เล่นจนตลาดปิด

**อ่านผลที่พิมพ์ออกมา** (ตัวเลขตัวอย่าง):
```
ส่งผลประเมิน 6 ล็อต · 201                                    ← ② ช่วงประเมิน
━━ รอบ R1 (1/8) · RFQ 8 รายการ · เสนอได้ 8 กระสอบ ━━         ← ④ รอบเปิด
   ส่ง 2 ข้อเสนอ 8 กระสอบ · คิด 0.0 วิ · 201
   R1-01 COMMERCIAL ← LOT-T01-2  4 กระสอบ × 132.29 บาท/กก. (x1.00 ของ TV)
   ...
   ผลรอบ R1:                                                 ← ③ หลังรอบปิด
     R1-01 COMMERCIAL ← LOT-T01-2  x1.00  ACCEPTED      ขายได้ 4/4 กระสอบ = 31,750 บาท
     R1-05 SPECIALTY  ← LOT-T01-3  x1.00  OUTBID        ขายได้ 0/4 กระสอบ
   รอบนี้ขายได้ 4 กระสอบ · 31,750 บาท
   สต็อก (เหลือ/ทั้งหมด): LOT-T01-1 10/10 · LOT-T01-2 6/10 · ... · รวมเหลือ 52/56 กระสอบ
   ขายแล้วรวม 31,750 บาท
```
`←` อ่านว่า "ขายล็อตนี้ให้ RFQ นี้" · ที่ขายไม่ได้ยังอยู่ในสต็อก ขายต่อรอบหน้าได้ · ความหมายของสถานะดูตารางในขั้นที่ 6 · ใช้ช่องเริ่มเกมใหม่นี้ทุกครั้งที่อยากเริ่มเกมใหม่ ก่อนรันลูปในข้อ 1.3 หรือข้อ 3

In [ ]:
r = api.post("/v1/sandbox/reset")      # เกมเดิมของทีมจบทันที ได้ล็อตชุดใหม่
s = r.json()
print(r.status_code, "ช่วง:", s["phase"], "· เหลือ", s["seconds_left"], "วินาที")

In [ ]:
run(lambda s: decide_rules(s, MARKUP), max_rounds=3)

**ลองเอง (5 นาที):** แก้ `MARKUP` ในข้อ 1.2 ให้แต่ละประเภทผู้ซื้อไม่เท่ากัน แล้วรันลูปอีกครั้ง — ผลเปลี่ยนอย่างไร

ดูสถานะ `ABOVE_LIMIT` (แพงเกินเพดาน) กับ `OUTBID` (แพ้ข้อเสนอที่คุ้มกว่า) เป็นตัวบอกทาง

## 2. เรียก LLM ครั้งแรก

ข้อ 1 สมองของ agent คือ**กฎ**ที่เราเขียนเอง ข้อ 2–3 จะให้ **LLM** (Large Language Model) ช่วยคิด ข้อนี้ฝึกเรียก LLM จาก Python ก่อน ยังไม่ยุ่งกับตลาด

**เรียก LLM หนึ่งครั้งเกิดอะไรขึ้น**

```
 notebook ──① ส่ง messages + ค่าตั้ง──▶ LLM gateway ของงาน ──② ส่งต่อ──▶ โมเดล event-llm
                                     (ตรวจ key ทีม                     (อ่าน messages ทั้งหมด
                                      จำกัดความถี่)                      แล้วเขียนคำตอบทีละ token)
 notebook ◀──③ คำตอบ + จำนวน token ที่ใช้─────────────────────────────────┘
```

- gateway พูด API แบบเดียวกับ OpenAI (OpenAI-compatible) จึงใช้ไลบรารี `openai` ได้เลย · ตัวแปร `llm` สร้างไว้แล้วในข้อ 0
- ใน `llm` ข้อ 0 ตั้ง `max_retries=0` = พลาดแล้ว**ไม่ลองซ้ำเอง** เพราะรอบในตลาดมีเวลาจำกัด เราจะตัดสินใจเองว่าพลาดแล้วทำอะไร (ข้อ 2.2)

**สิ่งที่ส่งไปในช่องข้างล่าง**

| ค่า | ใส่อะไร | แปลว่า |
|---|---|---|
| `model` | `LLM_MODEL` (`event-llm`) | ใช้โมเดลไหน · วันงานมีตัวเดียว |
| `messages` | รายการข้อความ แต่ละข้อมี `role` กับ `content` | บทสนทนาที่ส่งให้ LLM อ่าน · `role` = `"system"` (บทบาทและกติกา) หรือ `"user"` (คำถามของเรา) |
| `max_tokens` | `200` | คำตอบยาวได้สูงสุดกี่ token (token ≈ ชิ้นส่วนคำ) · เกินแล้วถูกตัดกลางประโยค |
| `reasoning_effort` | `"none"` | ปิดโหมดคิดก่อนตอบ (thinking) · เร็วกว่ามาก |

LLM จำอะไรข้ามครั้งไม่ได้: ทุกครั้งที่เรียก มันรู้แค่สิ่งที่อยู่ใน `messages` ครั้งนั้น

In [ ]:
t0 = time.time()
r = llm.chat.completions.create(
    model=LLM_MODEL, max_tokens=200, reasoning_effort="none",
    messages=[{"role": "user", "content": "อธิบายกาแฟโรบัสต้าใน 1 ประโยค"}])

print("คำตอบ:", r.choices[0].message.content)       # ข้อความที่ LLM ตอบ
print("จบแบบ:", r.choices[0].finish_reason)          # stop = ตอบจบเอง · length = ถูกตัดเพราะ max_tokens
print(f"ใช้เวลา {time.time() - t0:.1f} วิ · token ขาเข้า {r.usage.prompt_tokens} · ขาออก {r.usage.completion_tokens}")

**อ่านผลอย่างไร** (ตัวอย่าง)

```
คำตอบ: กาแฟโรบัสต้าเป็นกาแฟรสเข้ม ขม มีคาเฟอีนสูง ปลูกง่ายในที่ราบต่ำ ...
จบแบบ: stop
ใช้เวลา 1.4 วิ · token ขาเข้า 24 · ขาออก 41
```

| ส่วนของคำตอบ (`r`) | คืออะไร |
|---|---|
| `r.choices[0].message.content` | ข้อความที่ LLM ตอบ (เป็น **ข้อความ** เสมอ แม้จะขอ JSON ก็ตาม) |
| `r.choices[0].finish_reason` | `stop` = ตอบจบเอง · `length` = ถูกตัดเพราะถึง `max_tokens` (ถ้าเป็น JSON จะขาดกลางทาง อ่านไม่ได้) |
| `r.usage.prompt_tokens` | token ขาเข้า = ขนาดของ `messages` ที่เราส่งไป |
| `r.usage.completion_tokens` | token ขาออก = ความยาวของคำตอบ · ยิ่งยาวยิ่งช้า |

เวลาที่ใช้ขึ้นกับความยาวคำตอบเป็นหลัก และช้าลงเมื่อหลายทีมเรียกพร้อมกัน — ในตลาดหนึ่งรอบมี 90 วินาที LLM ต้องตอบให้ทันก่อนรอบปิด

**ลองเปลี่ยนค่า**: ช่องข้างล่างถามคำถามเดิมซ้ำ เพื่อดูผลของ `system`, `temperature` และ `max_tokens`

In [ ]:
def chat(user, system=None, temperature=0.7, max_tokens=200):
    """ถาม LLM หนึ่งครั้ง คืน (คำตอบ, finish_reason)"""
    messages = [{"role": "system", "content": system}] if system else []
    messages.append({"role": "user", "content": user})
    r = llm.chat.completions.create(model=LLM_MODEL, messages=messages, temperature=temperature,
                                    max_tokens=max_tokens, reasoning_effort="none")
    return r.choices[0].message.content.strip(), r.choices[0].finish_reason

QUESTION = "ตั้งชื่อร้านกาแฟโรบัสต้า 1 ชื่อ ตอบแค่ชื่อ"

# ① system prompt: บอกบทบาทและวิธีตอบ คำถามเดิมได้คำตอบคนละแบบ
print("① ไม่มี system   :", chat(QUESTION)[0])
print("① มี system      :", chat(QUESTION, system="คุณเป็นชาวไร่กาแฟที่ชุมพร ตั้งชื่อเป็นภาษาใต้")[0])

# ② temperature: 0 = ตอบเหมือนเดิมเกือบทุกครั้ง · สูง = หลากหลาย
print("② temperature 0  :", [chat(QUESTION, temperature=0)[0] for _ in range(3)])
print("② temperature 1.2:", [chat(QUESTION, temperature=1.2)[0] for _ in range(3)])

# ③ max_tokens น้อยเกิน: คำตอบถูกตัด finish_reason = length
text, finish = chat("อธิบายวิธีคั่วกาแฟ 5 ขั้นตอน", max_tokens=15)
print("③ max_tokens 15  :", repr(text), "· จบแบบ", finish)

**สิ่งที่ควรเห็น**

| ขั้น | ลองอะไร | ควรเห็น | ใช้กับ agent อย่างไร |
|---|---|---|---|
| ① | เพิ่ม `system` | ชื่อเปลี่ยนตามบทบาทที่บอก | ข้อ 3 ใช้ `system` บอกกติกาตลาดและหน้าที่ของ agent |
| ② | `temperature` 0 กับ 1.2 | 0 ได้ชื่อซ้ำ ๆ · 1.2 ได้ชื่อต่างกันทุกครั้ง | starter ใช้ 0.7: ไม่เหมือนกันทุกทีม แต่ไม่มั่วเกิน |
| ③ | `max_tokens` 15 | ประโยคขาดกลางทาง · `จบแบบ length` | ขอ JSON แล้วตั้งน้อยไป JSON จะขาด อ่านไม่ได้ |

แต่ละครั้งที่เรียกใช้เวลาและโควตาของทีม — ช่องนี้เรียก LLM 9 ครั้ง ไม่ต้องรันบ่อย · ถ้าเจอ `RateLimitError` (เรียกถี่เกินโควตา) รอสัก 1 นาทีแล้วรันใหม่

### 2.1 ขอคำตอบเป็น JSON ตาม schema

agent ต้องเอาคำตอบไป**คำนวณต่อ** (เช่น ตัวคูณราคา) โปรแกรมอ่านประโยคภาษาคนไม่ได้ จึงขอให้ LLM ตอบเป็น JSON ที่มีหน้าตาตามที่เรากำหนด

**ขั้นตอน**

1. เขียน **JSON schema** = แบบฟอร์มว่าคำตอบต้องมีช่องอะไร แต่ละช่องเป็นชนิดไหน
2. ส่ง schema ไปใน `response_format` · gateway บังคับให้โมเดลเขียนได้แค่ JSON ตามแบบนี้
3. คำตอบที่ได้ยังเป็น**ข้อความ** → แปลงเป็น dict ด้วย `json.loads()` แล้วใช้ได้เหมือน dict ทั่วไป

**`SCHEMA` ในช่องข้างล่าง** ขอคำตอบหน้าตาแบบนี้

```
{
  "markup": {"SPECIALTY": ตัวเลข, "COMMERCIAL": ตัวเลข, "INSTANT": ตัวเลข},   ← ตัวคูณราคาของผู้ซื้อแต่ละประเภท
  "reason": "ข้อความ"                                                    ← เหตุผล (ไว้อ่านตอน debug)
}
```

| ส่วนของ schema | แปลว่า |
|---|---|
| `"type": "object"` | คำตอบเป็น object (dict) |
| `"required": ["markup", "reason"]` | ต้องมีครบทั้งสองช่อง |
| `"properties": {...}` | แต่ละช่องเป็นอะไร: `markup` เป็น object ที่มี 3 ช่องเป็นตัวเลข (`number`) · `reason` เป็นข้อความ (`string`) |

ขอเหตุผล (`reason`) ไว้ด้วยเสมอ: ตอน LLM ตอบแปลก ๆ จะได้รู้ว่ามันคิดอะไร

In [ ]:
SCHEMA = {"type": "object", "required": ["markup", "reason"], "properties": {
    "markup": {"type": "object", "required": ["SPECIALTY", "COMMERCIAL", "INSTANT"], "properties": {
        "SPECIALTY": {"type": "number"}, "COMMERCIAL": {"type": "number"}, "INSTANT": {"type": "number"}}},
    "reason": {"type": "string"}}}

r = llm.chat.completions.create(
    model=LLM_MODEL, max_tokens=300, reasoning_effort="none",
    messages=[{"role": "user", "content": "Suggest a price markup (x true value) for each buyer type of green coffee. "
                                          "Specialty pays the most, Instant the least."}],
    response_format={"type": "json_schema", "json_schema": {"name": "answer", "schema": SCHEMA}})

text = r.choices[0].message.content           # ① ได้มาเป็นข้อความ
print("ข้อความที่ได้:", text)
answer = json.loads(text)                     # ② แปลงเป็น dict
print("หลัง json.loads:", type(answer).__name__)
print("markup:", answer["markup"])            # ③ ใช้เหมือน dict ทั่วไป
print("ตัวคูณของ INSTANT:", answer["markup"]["INSTANT"])
print("reason:", answer["reason"])

**อ่านผลอย่างไร** (ตัวเลขเป็นตัวอย่าง)

```
ข้อความที่ได้: {"markup": {"SPECIALTY": 1.4, "COMMERCIAL": 1.15, "INSTANT": 0.9}, "reason": "Specialty buyers ..."}
หลัง json.loads: dict
markup: {'SPECIALTY': 1.4, 'COMMERCIAL': 1.15, 'INSTANT': 0.9}
ตัวคูณของ INSTANT: 0.9
reason: Specialty buyers ...
```

- บรรทัดแรกเป็น**ข้อความ** (มีเครื่องหมาย `"` แบบ JSON บางครั้งขึ้นบรรทัดใหม่จัดย่อหน้ามาด้วย) · หลัง `json.loads` กลายเป็น `dict` จึงหยิบ `answer["markup"]["INSTANT"]` ได้
- prompt ข้อนี้เขียนเป็นภาษาอังกฤษ: LLM ส่วนใหญ่ทำตามคำสั่งภาษาอังกฤษได้แม่นกว่า และใช้ token น้อยกว่า

**สังเกต:** ได้ JSON ที่โปรแกรมอ่านได้ แต่ตัวเลขสมเหตุสมผลไหม? LLM ไม่รู้กติกาตลาดของเรา (ไม่รู้ TV ไม่รู้ผลรอบก่อน) จึงเดาจากความรู้ทั่วไป · รันซ้ำอาจได้ตัวเลขคนละชุด และบางครั้งได้ตัวเลขที่ใช้ไม่ได้จริง เช่น ตัวคูณ 5 เท่า
→ ต้อง (1) ให้ข้อมูลใน prompt ให้พอ (ข้อ 3.2) และ (2) ให้ Python ตรวจตัวเลขก่อนใช้เสมอ (ข้อ 3.3)

### 2.2 LLM พลาดได้เสมอ — agent ต้องไม่ตาย

ในตลาด ถ้าเรียก LLM แล้วโปรแกรมพัง agent จะหยุดส่งข้อเสนอทั้งเกม จึงต้องห่อการเรียก LLM ไว้ในฟังก์ชันที่ **ไม่ throw** (ไม่โยน error ออกมา) — พลาดเมื่อไรคืน `None` แล้วให้กฎทำงานแทน

**LLM พลาดได้แบบไหนบ้าง**

| พลาดแบบ | เห็นอะไร | เกิดจาก |
|---|---|---|
| ช้าเกิน | `APITimeoutError` | หลายทีมเรียกพร้อมกัน คำตอบยาว หรือเปิด thinking |
| เรียกถี่เกิน | `RateLimitError` (429) | ทีมเรียก LLM เกินโควตาที่ gateway กำหนด |
| key ผิด | `AuthenticationError` (401) | `LLM_KEY` ไม่ใช่ key ของทีม |
| ติดต่อไม่ได้ | `APIConnectionError` | ที่อยู่ gateway ผิด หรือเครือข่ายหลุด |
| JSON เสีย | `JSONDecodeError` | คำตอบถูกตัด (`max_tokens` น้อยไป) หรือโมเดลเขียนเพี้ยน |

**`ask_json()` ทำอะไรบ้าง** (ข้อ 3 และ starter ใช้ฟังก์ชันแบบนี้)

| ขั้น | ทำอะไร |
|---|---|
| ① | `llm.with_options(timeout=timeout)` ตั้งเวลารอ**เฉพาะครั้งนี้** · ในตลาดใส่เวลาที่เหลือในรอบ ลบเวลาที่ต้องใช้ส่งข้อเสนอ |
| ② | ส่ง `system` + `user` และขอ JSON ตาม `schema` · `temperature=0.7` · `max_tokens=500` เผื่อให้ JSON ไม่ขาด |
| ③ | `json.loads()` แปลงคำตอบเป็น dict แล้วคืนค่า |
| ④ | `except Exception` จับ**ทุก** error ในตารางข้างบน พิมพ์บอกว่าพลาดเพราะอะไร แล้วคืน `None` |

ช่องข้างล่างลอง 2 แบบ: (ก) ตั้ง timeout 0.5 วินาทีให้พลาดโดยตั้งใจ (ข) timeout ปกติ แล้วดูว่าโค้ดที่เรียกใช้รับมือกับ `None` อย่างไร

In [ ]:
def ask_json(system, user, schema, timeout=30):
    """ถาม LLM ขอ JSON ตาม schema · คืน dict หรือ None ถ้าพลาด (ไม่ throw)"""
    try:
        r = llm.with_options(timeout=timeout).chat.completions.create(       # ① เวลารอเฉพาะครั้งนี้
            model=LLM_MODEL, max_tokens=500, temperature=0.7, reasoning_effort="none",
            messages=[{"role": "system", "content": system}, {"role": "user", "content": user}],
            response_format={"type": "json_schema", "json_schema": {"name": "answer", "schema": schema}})  # ②
        return json.loads(r.choices[0].message.content)                     # ③
    except Exception as e:              # ④ timeout, 429, key ผิด, JSON เสีย ... อะไรก็ตาม
        print("LLM พลาด:", type(e).__name__, str(e)[:120])
        return None

FALLBACK = {"SPECIALTY": 1.00, "COMMERCIAL": 1.00, "INSTANT": 1.00}    # ทางสำรอง = กฎแบบข้อ 1

for name, timeout in [("(ก) timeout 0.5 วิ", 0.5), ("(ข) timeout 30 วิ", 30)]:
    t0 = time.time()
    answer = ask_json("You price green coffee. Answer JSON.", "Suggest markups for each buyer type.", SCHEMA,
                      timeout=timeout)
    if answer is None:
        chosen, source = FALLBACK, "กฎ (LLM พลาด)"
    else:
        chosen, source = answer["markup"], "LLM"
    print(f"{name}: ใช้ตัวคูณจาก {source} → {chosen} · {time.time() - t0:.1f} วิ\n")

**อ่านผลอย่างไร** (ตัวเลขเป็นตัวอย่าง)

```
LLM พลาด: APITimeoutError Request timed out.
(ก) timeout 0.5 วิ: ใช้ตัวคูณจาก กฎ (LLM พลาด) → {'SPECIALTY': 1.0, 'COMMERCIAL': 1.0, 'INSTANT': 1.0} · 0.5 วิ

(ข) timeout 30 วิ: ใช้ตัวคูณจาก LLM → {'SPECIALTY': 1.3, 'COMMERCIAL': 1.1, 'INSTANT': 0.95} · 1.8 วิ
```

- (ก) LLM ตอบไม่ทันใน 0.5 วินาที → `ask_json` พิมพ์ `LLM พลาด` แล้วคืน `None` → โปรแกรม**ไม่พัง** ใช้ตัวคูณสำรองแทน และเสียเวลาแค่ 0.5 วินาที
- (ข) LLM ตอบทัน → ใช้ตัวคูณจาก LLM · แต่ตัวเลขยัง**ไม่ได้ตรวจ** (อาจสลับลำดับประเภท หรือสูงเกินจริง) ข้อ 3.3 จะบังคับให้อยู่ในช่วงที่ปลอดภัยก่อนใช้
- รูปแบบ `answer = ask_json(...)` → `if answer is None: ใช้กฎ` คือหัวใจของข้อ 3: **agent ส่งข้อเสนอได้ทุกรอบ ไม่ว่า LLM จะเป็นอย่างไร**

**`llm.py` ใน starter ทำเพิ่มจากนี้**

- อ่าน JSON ที่ห่อมาใน ```` ``` ```` หรือมีส่วนคิด `<think>...</think>` นำหน้าได้ (`parse_json()`)
- จดสาเหตุที่พลาดไว้ใน `last_error` ให้ไปขึ้นใน log (`note: llm failed: ...` ข้อ 4.3)
- เปิด/ปิด thinking ได้จาก `.env` (`LLM_THINKING`)

**ลองเอง**: เปลี่ยน `timeout` ของ (ก) เป็น 2 หรือ 5 วินาที — LLM ตอบทันตั้งแต่กี่วินาที

## 3. ให้ LLM ช่วยตัดสินใจ

แบ่งงานให้ชัด: **LLM เลือกตัวคูณราคา** ของแต่ละประเภทผู้ซื้อ จากผลรอบก่อนๆ · **Python คุมตัวเลข** (ล็อต กระสอบ สเปก กรอบตัวคูณ)

ลูปยังเป็น `run()` ตัวเดิมจากข้อ 1.3 **เปลี่ยนแค่สมอง** จาก `decide_rules` เป็น `decide_llm` · ในแต่ละรอบ `decide_llm` ทำแบบนี้:

```
รอบเปิด → run() เรียก decide_llm(s)
          ① สร้างข้อความ user: รอบที่เท่าไร + ตัวคูณตอนนี้ + ผล 5 รอบล่าสุดจาก history
          ② ส่ง SYSTEM + user ให้ LLM ขอคำตอบเป็น JSON (ask_json จากข้อ 2.2)
          ③ LLM ตอบทัน → ตรวจตัวคูณ บีบให้อยู่ในกรอบ 0.85–1.30 แล้วจำไว้ใช้รอบหน้า
          ④ LLM พลาด → ใช้ตัวคูณเดิม
          ⑤ เรียก decide_rules(s, ตัวคูณ) จากข้อ 1.2 → ได้ข้อเสนอ → run() ส่งให้
```

LLM ไม่ได้เลือก RFQ ล็อต หรือจำนวนกระสอบเอง เลือกแค่ตัวคูณ 3 ตัว ที่เหลือเป็นกฎเดิมทั้งหมด

### 3.1 SYSTEM: คำสั่งคงที่ ส่งเหมือนเดิมทุกรอบ

prompt ที่ส่งให้ LLM มี 2 ส่วน (สไลด์ "prompt มี 2 ส่วน"):
- **system** = บทบาทและกติกา เขียนครั้งเดียว ไม่เปลี่ยนตามรอบ (ช่องนี้)
- **user** = ข้อมูลของรอบนี้ สร้างใหม่ทุกรอบ (ข้อ 3.2)

ทีละประโยคใน `SYSTEM`:

| ประโยค | บอก LLM ว่า |
|---|---|
| You set prices … sealed-offer market | บทบาท: ตั้งราคาให้ฟาร์ม ในตลาดยื่นซองปิด (ไม่เห็นราคาคนอื่น) |
| price = markup x the lot's true value | ราคา = ตัวคูณ × TV สิ่งที่ LLM เลือกคือตัวคูณ |
| Buyers take the cheapest … hidden ceiling | ผู้ซื้อซื้อข้อเสนอที่คุ้มสุดก่อน และไม่รับราคาเกินเพดานที่ซ่อนไว้ |
| Specialty usually pays the most … | ความรู้ตั้งต้น: Specialty จ่ายมากสุด Instant น้อยสุด |
| ABOVE_LIMIT = … ACCEPTED = sold | ความหมายของสถานะในผลรอบ ให้อ่าน history ออก |
| Unsold beans are worth only 0.80 … | ขายไม่ออก = เสียมูลค่า จึงไม่ควรตั้งแพงจนขายไม่ได้ |
| Answer only JSON … | ตอบ JSON เท่านั้น เหตุผลสั้นๆ (คำตอบสั้น = เร็ว และไม่เกิน `max_tokens`) |

เขียนภาษาอังกฤษ เพราะใช้ token น้อยกว่าภาษาไทย และโมเดลเล็กทำตามคำสั่งภาษาอังกฤษได้แม่นกว่า

In [ ]:
SYSTEM = """You set prices for a coffee farm in a sealed-offer market.
price = markup x the lot's true value. Buyers take the cheapest offers (price / true value) first
and reject any price above their hidden ceiling. Specialty usually pays the most, Instant the least.
ABOVE_LIMIT = too expensive for that buyer, OUTBID = another seller was cheaper, ACCEPTED = sold.
Unsold beans are worth only 0.80 x true value at the end.
Answer only JSON; keep the reason to one short sentence."""

print(SYSTEM)

### 3.2 user: ข้อมูลของรอบนี้ สร้างใหม่ทุกรอบ

LLM **จำอะไรเองไม่ได้** แต่ละครั้งที่เรียกเริ่มจากศูนย์ (สไลด์ "LLM ทำงานอย่างไร") สิ่งที่อยากให้รู้ต้องใส่ในข้อความทุกครั้ง:

| บรรทัด | มาจาก |
|---|---|
| `Round R3 of 8.` | `state["round"]` · ให้รู้ว่าใกล้จบเกมแค่ไหน |
| `Current markups: {...}` | ตัวแปร `markup` ตัวคูณที่ใช้อยู่ |
| `My recent results:` + 5 บรรทัด | `history[-5:]` ที่ `show_results()` เก็บไว้ในข้อ 1.3 นี่คือ **feedback loop** |
| `Choose the markups for this round.` | คำสั่ง |

ส่งแค่ 5 รอบล่าสุด ไม่ส่งทั้งหมด เพื่อให้ prompt สั้น เร็ว และไม่เกินขนาด context (สไลด์ "context")

ช่องข้างล่างสร้างฟังก์ชัน `user_prompt()` แล้วพิมพ์ให้ดูว่า LLM จะเห็นอะไร (ถ้าเพิ่งรันข้อ 1.3 จะเห็นผลรอบใน history แล้ว)

In [ ]:
markup = dict(MARKUP)         # ตัวคูณที่ LLM ปรับ · คัดลอกจาก MARKUP ในข้อ 1.2 จะได้ไม่แก้ของเดิม

def user_prompt(state):
    """ข้อความ user ของรอบนี้: รอบ + ตัวคูณตอนนี้ + ผล 5 รอบล่าสุด"""
    rnd = state["round"]
    return "\n".join([
        f"Round {rnd['round_id']} of {rnd['of']}." if rnd else "Round: not open yet.",
        f"Current markups: {json.dumps(markup)}",
        "My recent results:", *(history[-5:] or ["none yet"]),
        "Choose the markups for this round."])

print(user_prompt(api.get("/v1/state").json()))

### 3.3 `decide_llm(state)`: สมองแบบ LLM + กฎ

ฟังก์ชันนี้รับสถานะ `state` อย่างเดียวแล้วคืนรายการข้อเสนอ เหมือน `decide_rules` จึงส่งให้ `run()` ได้ตรงๆ (ไม่ต้องใช้ `lambda`)

| | ในโค้ด | ทำไม |
|---|---|---|
| ① | `user = user_prompt(state)` | ข้อมูลของรอบนี้ (ข้อ 3.2) |
| ② | `ask_json(SYSTEM, user, SCHEMA, timeout=...)` | ถาม LLM ขอ JSON ตาม `SCHEMA` (ข้อ 2.1) · `ask_json` ไม่ throw พลาดเมื่อไรคืน `None` (ข้อ 2.2) · timeout = เวลาที่เหลือในรอบ − 10 วิ (อย่างน้อย 3 วิ) เผื่อเวลาส่งข้อเสนอ |
| ③ | `min(max(ค่า, LOW), HIGH)` | **guard:** LLM ตอบ 5.0 หรือ 0.1 ก็ถูกบีบให้อยู่ใน 0.85–1.30 · ค่าไหนไม่ใช่ตัวเลข ใช้ค่าเดิม · ค่าที่ได้เก็บใน `markup` จึงใช้ต่อรอบหน้า |
| ④ | `else` | LLM พลาด (ช้า ล่ม JSON เสีย 429) → ตัวคูณเดิม agent ไม่หยุด |
| ⑤ | `decide_rules(state, markup)` | กฎเดิมจากข้อ 1.2 เลือก RFQ ล็อต กระสอบ และคิดราคาด้วยตัวคูณใหม่ |

In [ ]:
LOW, HIGH = 0.85, 1.30        # กรอบที่ Python ไม่ยอมให้ LLM ออกนอก

def decide_llm(state):
    # ① ข้อมูลของรอบนี้
    user = user_prompt(state)
    # ② ถาม LLM · เผื่อเวลาส่งข้อเสนอ 10 วินาที
    answer = ask_json(SYSTEM, user, SCHEMA, timeout=max(3, (state["seconds_left"] or 30) - 10))
    if answer:
        # ③ ตรวจและบีบตัวคูณให้อยู่ในกรอบ แล้วจำไว้ใน markup
        for t in markup:
            try:
                markup[t] = round(min(max(float(answer["markup"][t]), LOW), HIGH), 3)
            except (KeyError, TypeError, ValueError):
                pass                   # ค่าไหนแปลกก็ใช้ค่าเดิม
        print("   LLM:", markup, "·", str(answer.get("reason"))[:120])
    else:
        # ④ LLM พลาด: ใช้ตัวคูณเดิม
        print("   ใช้กฎ: ตัวคูณเดิม", markup)
    # ⑤ กฎเดิมเลือกข้อเสนอด้วยตัวคูณนี้
    return decide_rules(state, markup)

### 3.4 เล่นด้วยสมอง LLM

**รันสองช่องข้างล่างตามลำดับ:** เริ่มเกมใหม่ → ลูป · หยุดได้ด้วยปุ่ม ■

- ช่องเริ่มเกมใหม่ล้าง `history` ด้วย ให้ LLM เห็นเฉพาะผลของเกมนี้ (ล็อตของเกมก่อนเป็นคนละชุด)
- `run(decide_llm, max_rounds=3)` = ลูปเดิม ใช้ `decide_llm` เป็นสมอง ขาย 3 รอบแล้วหยุด

**อ่านผลที่พิมพ์ออกมา** (ตัวเลขตัวอย่าง) ส่วนที่ต่างจากข้อ 1.3 คือบรรทัดของ LLM:
```
━━ รอบ R2 (2/8) · RFQ 8 รายการ · เสนอได้ 8 กระสอบ ━━
   LLM: {'SPECIALTY': 1.1, 'COMMERCIAL': 1.0, 'INSTANT': 0.9} · Commercial sold out, ...   ← ③ LLM ตอบทัน
   ส่ง 3 ข้อเสนอ 8 กระสอบ · คิด 2.4 วิ · 201                                            ← เวลาคิดรวมเวลาของ LLM
   ...
━━ รอบ R3 (3/8) · ... ━━
LLM พลาด: APITimeoutError Request timed out.                                           ← ④ LLM พลาด
   ใช้กฎ: ตัวคูณเดิม {'SPECIALTY': 1.1, 'COMMERCIAL': 1.0, 'INSTANT': 0.9}
```
ดูว่าเหตุผลของ LLM ตรงกับผลรอบก่อนไหม เช่น ถ้าเห็น `ABOVE_LIMIT` ควรลดตัวคูณของประเภทนั้น

In [ ]:
r = api.post("/v1/sandbox/reset")      # เกมเดิมของทีมจบทันที ได้ล็อตชุดใหม่
history.clear()                        # ล้างผลของเกมก่อน ให้ LLM เห็นเฉพาะเกมนี้
markup = dict(MARKUP)                  # เริ่มตัวคูณใหม่จาก MARKUP ในข้อ 1.2
s = r.json()
print(r.status_code, phase_line(s), "· ได้", len(s["lots"]), "ล็อต")

In [ ]:
run(decide_llm, max_rounds=3)

**ลองเอง:** แก้ทีละอย่าง แล้วรันสองช่องข้างบนใหม่ เทียบผลกับข้อ 1.3
- แก้ `SYSTEM` (ข้อ 3.1) เช่น บอกให้ลดตัวคูณเมื่อเห็น `ABOVE_LIMIT`
- เปลี่ยน `history[-5:]` ใน `user_prompt` (ข้อ 3.2) เป็น 3 หรือ 8 รอบ
- ขยายหรือแคบกรอบ `LOW, HIGH` (ข้อ 3.3)

**คุยกัน:** LLM ปรับตัวคูณตามผลจริงไหม · ช้าแค่ไหน · ถ้า 55 ทีมเรียกพร้อมกันตอนรอบเปิดจะเป็นอย่างไร

## 4. starter agent

**starter** คือ agent ที่ทีมงานเขียนไว้ให้ ทำทุกอย่างที่เราทำในข้อ 1–3 ครบ และเล่นจนตลาดปิดได้ทันที — บ่ายนี้ทีมเอา starter ไปแก้ต่อ ไม่ต้องเขียนใหม่ทั้งหมด

starter อยู่ในโฟลเดอร์บนสุดของ repo (ที่เดียวกับ `README.md`) ส่วน notebook นี้อยู่ใน `notebooks/`

```
Coffee-TESA-starter/
├── main.py        ← ลูปหลัก (เหมือน run() ข้อ 1.3)
├── strategy.py    ← สมองของ agent  ← ไฟล์ที่ทีมควรแก้
├── appraisal.py   ← สูตร TV และใบประเมิน (เหมือนข้อ 1.1)
├── llm.py         ← เรียก LLM ขอ JSON (เหมือน ask_json ข้อ 2)
├── memory.py      ← จำผลรอบก่อน ๆ (เหมือน history ข้อ 1.3)
├── market.py      ← เรียก API ตลาด
├── .env.example   ← ตัวอย่างไฟล์ตั้งค่า
├── logs/          ← starter สร้างเองตอนรัน
└── notebooks/     ← notebook นี้
```

**ของใน notebook ↔ ของใน starter**

| ใน notebook | ใน starter | ต่างกันอย่างไร |
|---|---|---|
| `true_value()`, `spec_ok()` (ข้อ 1.1) | `appraisal.py`, `strategy.spec_ok()` | สูตรเดียวกัน |
| `run()` (ข้อ 1.3) | `main.py` → `play()` | เล่นจนตลาดปิด · `--forever` เล่นเกมใหม่ต่อเอง · network ล่มก็ไม่หยุด |
| `decide_rules()` (ข้อ 1.2) | `strategy.rule_plan()` | ไม่ได้ขายล็อตละ 1 กระสอบ แต่**กระจายของให้หมดทันรอบสุดท้าย** (`sell_budget()`) |
| `decide_llm()` (ข้อ 3) | `strategy.llm_plan()` | LLM ไม่ได้เลือกแค่ตัวคูณ แต่**เลือกเองว่าเสนอล็อตไหนให้ RFQ ไหน กี่กระสอบ ตัวคูณเท่าไร** |
| ช่วงตัวคูณ `LOW`–`HIGH` (ข้อ 3.3) | `strategy.guard()` | ตรวจ**ทุก**ข้อเสนอก่อนส่ง ไม่ว่ามาจาก LLM หรือกฎ |
| `history` (ข้อ 1.3) | `memory.py` | บันทึกลงไฟล์ ปิดโปรแกรมแล้วเปิดใหม่ก็ยังจำได้ |
| ข้อความที่ print | `logs/decisions.jsonl` | บันทึกทุกการตัดสินใจไว้อ่านทีหลัง (ข้อ 4.3) |

ข้อ 4 มี 4 ขั้น: **4.1** ติดตั้งและรัน → **4.2** อ่านโค้ดว่าหนึ่งรอบทำอะไร → **4.3** อ่าน log ว่าขายอะไรได้ → **4.4** แก้ 1 จุดแล้วเทียบผล

### 4.1 ติดตั้งและรัน starter

starter รันใน **Terminal** (ไม่ได้รันใน notebook) เปิด Terminal ใน VS Code (เมนู Terminal → New Terminal) ให้อยู่ที่โฟลเดอร์บนสุดของ repo — โฟลเดอร์ที่มี `main.py` · ไม่แน่ใจให้พิมพ์ `dir` แล้วดูว่ามี `main.py`

**ขั้นที่ 1 สร้าง venv และติดตั้ง library** (ทำครั้งเดียว · คำสั่งของ Windows)

```
python -m venv .venv
.venv\Scripts\pip install -r requirements.txt
```

**ขั้นที่ 2 สร้างไฟล์ตั้งค่า** (ทำครั้งเดียว)

```
copy .env.example .env
```

แล้วเปิด `.env` ใน VS Code แก้ค่าตามตาราง (ใช้ค่าเดียวกับช่องข้อ 0)

| ตัวแปร | ใส่อะไร | ข้อ 0 ใช้ชื่อ |
|---|---|---|
| `MARKET_URL` | ที่อยู่ตลาดฝึก | `MARKET` |
| `TEAM_KEY` | key ของทีม (ขึ้นต้นด้วย `sk-T`) | `KEY` |
| `USE_LLM` | `1` = ใช้ LLM · `0` = ใช้กฎอย่างเดียว | — |
| `LLM_BASE_URL` | ที่อยู่ LLM gateway | `LLM_URL` |
| `LLM_MODEL` | `event-llm` (ไม่ต้องแก้) | `LLM_MODEL` |
| `LLM_API_KEY` | key เดียวกับ `TEAM_KEY` | `LLM_KEY` |
| `LLM_TIMEOUT` | รอ LLM ได้นานสุดกี่วินาที (`60`) · starter ตัดให้สั้นลงเองถ้ารอบใกล้หมด | — |
| `LLM_THINKING` | `0` = ปิด thinking (เร็ว) · `1` = เปิด (ช้า กิน token) | — |

`.env` มี key ของทีม — **ห้าม commit ขึ้น GitHub** (repo ตั้ง `.gitignore` ไว้ให้แล้ว)

**ขั้นที่ 3 รัน**

```
.venv\Scripts\python main.py --forever
```

- ไม่ใส่ `--forever` = เล่นเกมเดียวแล้วหยุดเมื่อตลาดปิด (ใช้วันแข่ง)
- ใส่ `--forever` = จบเกมแล้วรอเกมใหม่ แล้วเล่นต่อเอง (ใช้กับตลาดฝึก ซึ่งเริ่มเกมใหม่เองหลังปิด 5 นาที)
- หยุดด้วย **Ctrl+C** · รันใหม่กลางเกมได้ starter อ่านสถานะจากตลาดแล้วเล่นต่อ

**ระวัง:** starter กับ notebook นี้ใช้ key เดียวกัน = **ตลาดเดียวกัน** ระหว่างที่ starter เล่นอยู่ อย่ารันช่องในข้อ 1–3 โดยเฉพาะช่องที่เริ่มเกมใหม่ (`/v1/sandbox/reset`) ไม่งั้นเกมของ starter จะถูกล้าง

**สิ่งที่ starter พิมพ์ออกมา**

แต่ละบรรทัด = เวลา ชื่อเหตุการณ์ และรายละเอียดสั้น ๆ (ตัวเลขข้างล่างเป็นตัวอย่าง)

```
10:02:05 new_game {"market": "sandbox", "lots": 6, "tv": {"LOT-T07-1": 112.4, "LOT-T07-2": 98.1, ...}}
10:02:07 appraisal_sent {"tv": {"LOT-T07-1": 112.4, ...}}
10:03:09 offers_sent {"round": "R1", "source": "llm", "n": 5, "kg": 600, "seconds": 8.4}
10:04:45 offers_sent {"round": "R2", "source": "llm+rules (3+2)", "n": 5, "kg": 600, "seconds": 9.1}
...
10:16:30 market_closed {"standings": {"revenue_thb": ..., "kg_sold": ..., "kg_remaining": ...}}
```

| เหตุการณ์ | แปลว่า |
|---|---|
| `new_game` | เจอเกมใหม่ · `lots` = มีกี่ล็อต · `tv` = TV ที่ประเมินจากผลวัดตัวอย่างของแต่ละล็อต |
| `appraisal_sent` | ส่งใบประเมินแล้ว (ช่วง APPRAISAL) |
| `offers_sent` | ส่งข้อเสนอของรอบนี้แล้ว · `n` = กี่ข้อเสนอ · `kg` = รวมกี่กก. · `seconds` = ใช้เวลาคิดกี่วินาที · `source` ดูข้อ 4.3 |
| `market_closed` | ตลาดปิด · `standings` = ขายได้กี่บาท กี่กก. เหลือกี่กก. |
| `batch_rejected`, `offer_rejected` | ตลาดไม่รับข้อเสนอ · starter ส่งทีละข้อแทนแล้วข้ามข้อที่ผิด · อ่าน `error` ว่าผิดเพราะอะไร |
| `strategy_crashed` | โค้ดใน `strategy.py` พัง (มักเกิดหลังเราแก้โค้ด) · รอบนั้น starter ใช้กฎแทนเพื่อไม่ให้เสียรอบ · **รีบแก้** |
| `round_already_sent` | รันใหม่กลางรอบที่ส่งไปแล้ว · ไม่ส่งซ้ำ |
| `state_error` | ติดต่อตลาดไม่ได้ · starter รอ 3 วินาทีแล้วลองใหม่เอง |

ปล่อย starter เล่นให้จบหนึ่งเกม (ประมาณ 15 นาที) ระหว่างรอ อ่านข้อ 4.2

### 4.2 อ่านโค้ด starter: หนึ่งรอบทำอะไรบ้าง

**`main.py` → `play()`** วนอ่านสถานะตลาดทุก ≤ 5 วินาที แล้วทำตามช่วงของตลาด เหมือน `run()` ข้อ 1.3 แต่ครบทุกช่วง

```
            GET /v1/state
                 │
  ล็อตเปลี่ยน? ──┼──▶ ① เกมใหม่: ประเมิน TV (appraisal.py) · เปิดไฟล์ memory ใหม่
                 │
     APPRAISAL ──┼──▶ ② ส่งใบประเมิน (ครั้งเดียว)
                 │
    ROUND_OPEN ──┼──▶ ③ จำผลรอบก่อน (memory.record_results)
   (รอบใหม่)     │    ④ strategy.decide()  ← สมอง
                 │    ⑤ ส่งข้อเสนอ · จำว่าส่งอะไร · เขียน log offers_sent
                 │
        CLOSED ──┴──▶ ⑥ จำผลรอบสุดท้าย · เขียน log market_closed · ไม่มี --forever ก็หยุด
```

**`strategy.py` → `decide()`** คือขั้น ④ ข้างบน ทำ 4 อย่างตามลำดับ

| ขั้น | ฟังก์ชัน | ทำอะไร |
|---|---|---|
| 1 | `sell_budget()` | รอบนี้ควรเสนอแต่ละล็อตกี่กก. = ของที่เหลือ ÷ รอบที่เหลือ × `PACE` (1.3) · ขายช้าไป ของค้างจะเหลือแค่ 0.80 × TV |
| 2 | `candidates()` | จับคู่ (ล็อต, RFQ) ทุกคู่ที่ตัวอย่างผ่านสเปก แล้วสลับลำดับแบบสุ่ม (RFQ ใหญ่มีโอกาสขึ้นก่อน) · ทุกทีมไม่แห่ไป RFQ แรกพร้อมกัน |
| 3 | `llm_plan()` | ส่งตารางคู่ให้ LLM เลือก: คู่ไหน กี่กระสอบ ตัวคูณเท่าไร · ถ้า LLM ช้า ปิดอยู่ หรือตอบมั่ว → ใช้ `rule_plan()` (ล็อต TV สูงก่อน ที่ตัวคูณ `MARKUP`) · ถ้า LLM เสนอน้อยไป กฎเติมให้ครบ |
| 4 | `guard()` | ตรวจทุกข้อเสนอ: ล็อตและ RFQ มีจริง · ไม่ซ้ำคู่ · ผ่านสเปก · กก. เป็นทวีคูณ 60 ไม่เกินของที่เหลือ ไม่เกินที่ RFQ ขอ ไม่เกินโควตารอบ · ตัวคูณอยู่ใน 0.85–1.30 · แล้วคิดราคา = ตัวคูณ × TV |

ทำไมต้องมี `guard()`: LLM ตอบตัวเลขผิดได้เสมอ (กระสอบเกิน ตัวคูณ 5.0 คู่ที่ไม่มีจริง) — **ห้ามส่งตัวเลขของ LLM ไปตรง ๆ** เหมือนช่วง `LOW`–`HIGH` ในข้อ 3.3

**อยากเปลี่ยนอะไร แก้ตรงไหน** (ในโค้ดมี `TODO` บอกไว้ทุกจุด)

| อยากให้ agent… | แก้ที่ |
|---|---|
| ตั้งราคาต่างกันตามประเภทผู้ซื้อ | `MARKUP` บนสุดของ `strategy.py` (ตอนนี้ 1.00 ทุกประเภท) — **จุดนี้ได้ผลมากที่สุด** |
| ขยับราคาตามผลรอบก่อน (ขายได้ → ขึ้น · แพ้/เกินเพดาน → ลง) | `rule_plan()` และ `memory.py` |
| ไม่โดน `SPEC_MISMATCH` | `spec_ok()` เว้นระยะจากขอบสเปก เพราะผลวัดตัวอย่างคลาดเคลื่อน |
| ขายมาก/น้อยตามปฏิทินตลาด | `sell_budget()` |
| ให้ LLM มีข้อมูลคิดมากขึ้น | `SYSTEM` และข้อความ `user` ใน `llm_plan()` · `memory.summary()` |
| ประเมินราคาแม่นขึ้น | `estimate_lots()` ใน `appraisal.py` |

ลำดับอ่านโค้ดที่แนะนำ: `main.py` (`play()`) → `strategy.py` (`decide()` แล้วไล่ขึ้นไปทีละฟังก์ชัน) → `guard()`

### 4.3 อ่าน log ของ starter: ขายอะไรได้บ้าง

starter เขียนทุกเหตุการณ์ลง `logs/decisions.jsonl` บรรทัดละหนึ่ง JSON (บรรทัดเดียวกับที่พิมพ์ใน Terminal แต่ครบกว่า) และเขียนผลแต่ละรอบลง `logs/memory-*.json` (หนึ่งไฟล์ต่อเกม)

บรรทัด `offers_sent` มีช่องที่ Terminal ไม่ได้พิมพ์ให้ดู

| ช่อง | คืออะไร |
|---|---|
| `offers` | ข้อเสนอที่ส่งจริงทุกข้อ (`rfq_id`, `lot_id`, `kg`, `price_per_kg`) |
| `notes` | starter จดไว้ว่าเกิดอะไรตอนคิด เช่น `llm reason: …` (เหตุผลที่ LLM ให้) · `llm failed: …` (LLM พลาดเพราะอะไร) · `guard dropped …` (guard ทิ้งข้อเสนอไหน) |
| `source` | ข้อเสนอมาจากไหน (ตารางข้างล่าง) |

| `source` | แปลว่า |
|---|---|
| `llm` | LLM เลือกทั้งหมด |
| `llm+rules (3+2)` | LLM เลือก 3 ข้อ กฎเติมอีก 2 ข้อให้ครบปริมาณ |
| `rules` | ใช้กฎอย่างเดียว: LLM ปิดอยู่ ช้า หรือตอบใช้ไม่ได้ (ดูสาเหตุใน `notes`) |
| `none` | ไม่มีล็อตไหนผ่านสเปกของ RFQ รอบนี้ หรือขายหมดแล้ว |

ช่องข้างล่างอ่านทั้งสองไฟล์ แล้วแสดง**เกมล่าสุด**ทีละรอบ: ส่งอะไรไป ผลเป็นอย่างไร ขายได้กี่บาท — รันได้ทั้งระหว่างที่ starter เล่นอยู่และหลังจบเกม

In [ ]:
from pathlib import Path

# ① หาโฟลเดอร์ starter: notebook อยู่ใน notebooks/ ส่วน main.py อยู่ชั้นบน
STARTER = next((p for p in (Path(".."), Path("../../starter")) if (p / "main.py").exists()), Path(".."))
LOGS = STARTER / "logs"

# ② อ่าน decisions.jsonl แล้วตัดเอาเฉพาะเกมล่าสุด (ตั้งแต่ new_game ครั้งสุดท้าย)
log_file = LOGS / "decisions.jsonl"
rows = [json.loads(line) for line in log_file.read_text(encoding="utf-8").splitlines()] if log_file.exists() else []
starts = [i for i, row in enumerate(rows) if row["event"] == "new_game"]
game = rows[starts[-1]:] if starts else []

# ③ อ่านผลแต่ละรอบจากไฟล์ memory ล่าสุด: sent = ที่ส่งไป (มีประเภทผู้ซื้อ ตัวคูณ) · results = ผลหลังรอบปิด
mem_files = sorted(LOGS.glob("memory-*.json"), key=lambda p: p.stat().st_mtime)
mem = json.loads(mem_files[-1].read_text(encoding="utf-8")) if mem_files else {"sent": {}, "rounds": []}
results = {rnd["round_id"]: {(o["rfq_id"], o["lot_id"]): o for o in rnd["my_offers"]} for rnd in mem["rounds"]}

if not game:
    print("ยังไม่มี log: รัน starter ตามข้อ 4.1 ก่อน · ช่องนี้หา log ที่", log_file.resolve())
else:
    tv = game[0]["tv"]
    print(f"เกมล่าสุดเริ่ม {game[0]['time'][11:]} · {game[0]['lots']} ล็อต · TV ที่ประเมิน:",
          ", ".join(f"{lot} {v:.2f}" for lot, v in tv.items()))
    total_bags = total_baht = 0
    for row in game[1:]:
        # ④ เหตุการณ์อื่นที่ไม่ใช่การส่งข้อเสนอ
        if row["event"] == "appraisal_sent":
            print(f"\n{row['time'][11:]} ส่งใบประเมินแล้ว")
            continue
        if row["event"] == "market_closed":
            st = row["standings"]
            print(f"\n{row['time'][11:]} ตลาดปิด · ตลาดนับว่าขายได้ {st['kg_sold'] // BAG} กระสอบ"
                  f" {st['revenue_thb']:,.0f} บาท · เหลือ {st['kg_remaining'] // BAG} จาก {st['kg_total'] // BAG} กระสอบ")
            continue
        if row["event"] != "offers_sent":                  # error ต่าง ๆ: อ่าน error ว่าผิดเพราะอะไร
            detail = {k: v for k, v in row.items() if k not in ("time", "event")}
            print(f"\n{row['time'][11:]} {row['event']} {json.dumps(detail, ensure_ascii=False)[:150]}")
            continue
        # ⑤ หนึ่งรอบ: หัวรอบ → ข้อเสนอแต่ละข้อกับผล → notes
        rid = row["round"]
        print(f"\n── {rid} · {row['time'][11:]} · มาจาก {row['source']} · {row['n']} ข้อเสนอ "
              f"{row['kg'] // BAG} กระสอบ · คิด {row['seconds']} วิ")
        sent = {(o["rfq_id"], o["lot_id"]): o for o in mem["sent"].get(rid, [])}
        for o in row["offers"]:
            key = (o["rfq_id"], o["lot_id"])
            buyer_type = sent.get(key, {}).get("buyer_type", "?")
            markup = o["price_per_kg"] / tv[o["lot_id"]]
            res = results.get(rid, {}).get(key)
            if res is None:
                outcome = "รอผล (รอบยังไม่ปิด)"
            else:
                sold = res["filled_kg"] // BAG
                baht = res["filled_kg"] * res["price_per_kg"]
                total_bags, total_baht = total_bags + sold, total_baht + baht
                outcome = f"{res['status']:<13} ขายได้ {sold} กระสอบ {baht:>9,.0f} บาท"
            print(f"   {o['rfq_id']:<7} {buyer_type:<10} {o['lot_id']}  {o['kg'] // BAG} กระสอบ"
                  f"  {o['price_per_kg']:>7.2f} บาท/กก. (×{markup:.2f})  → {outcome}")
        for note in row["notes"]:
            print("     note:", note[:150])
    print(f"\nรวมทั้งเกมถึงตอนนี้: ขายได้ {total_bags} กระสอบ {total_baht:,.0f} บาท")

**อ่านผลอย่างไร** (ตัวเลขเป็นตัวอย่าง)

```
── R2 · 10:04:45 · มาจาก llm+rules (3+2) · 5 ข้อเสนอ 10 กระสอบ · คิด 9.1 วิ
   R2-01   SPECIALTY  LOT-T07-1  2 กระสอบ   123.64 บาท/กก. (×1.10)  → ACCEPTED      ขายได้ 2 กระสอบ    14,837 บาท
   R2-04   INSTANT    LOT-T07-3  3 กระสอบ    93.20 บาท/กก. (×1.00)  → OUTBID        ขายได้ 0 กระสอบ         0 บาท
     note: llm reason: Specialty accepted last round, ask a bit more ...
```

- `── R2 …` หัวรอบ: ข้อเสนอมาจากไหน ส่งกี่ข้อ กี่กระสอบ LLM คิดนานเท่าไร
- แต่ละบรรทัด = 1 ข้อเสนอ: RFQ · ประเภทผู้ซื้อ · ล็อต · จำนวน · ราคา (ตัวคูณ = ราคา ÷ TV ที่ประเมิน) · **ผล**
- `note:` สิ่งที่ starter จดไว้ เช่น เหตุผลของ LLM
- บรรทัดสุดท้าย: รวมขายได้กี่กระสอบ กี่บาท

**ผลแต่ละแบบบอกอะไร แล้วควรทำอะไร**

| ผล | แปลว่า | ลองทำ |
|---|---|---|
| `ACCEPTED` | ขายได้ครบ | ผู้ซื้อประเภทนี้อาจจ่ายได้มากกว่านี้ → ลองขึ้นตัวคูณนิดหนึ่ง |
| `PARTIAL` | ขายได้บางส่วน | ผู้ซื้อซื้อครบที่ขอแล้ว จากคนที่ถูกกว่าก่อน |
| `OUTBID` | แพ้ทีมอื่นที่เสนอคุ้มกว่า | ลดตัวคูณ หรือไปเสนอ RFQ อื่น |
| `ABOVE_LIMIT` | แพงเกินเพดานที่ผู้ซื้อรายนี้รับได้ | ลดตัวคูณของประเภทนี้ |
| `SPEC_MISMATCH` | ของจริงไม่ผ่านสเปก (ตัวอย่างหลอกตา) | เว้นระยะจากขอบสเปกใน `spec_ok()` |

ถ้าเห็น `มาจาก rules` บ่อย ให้ดู `note: llm failed …` ว่า LLM พลาดเพราะอะไร (ช้าเกินรอบ · ตอบไม่เป็น JSON · key ผิด)

### 4.4 แก้ 1 จุดแล้วเทียบผล

ทดลองแบบนักวิทยาศาสตร์: **แก้ทีละอย่าง เล่นหลายเกม แล้วเทียบตัวเลข** — แก้หลายอย่างพร้อมกันจะไม่รู้ว่าอะไรทำให้ดีขึ้นหรือแย่ลง

1. เล่นเกมแรกด้วย starter เดิม (ข้อ 4.1) ให้จบ แล้วรันช่องข้างล่าง โดยตั้ง `CHANGE = "starter เดิม"`
2. เปิด `strategy.py` แก้ **1 จุด** เช่น `MARKUP` ของผู้ซื้อประเภทเดียว
3. หยุด starter (Ctrl+C) แล้วรันใหม่ (starter อ่านโค้ดใหม่ตอนเริ่มเท่านั้น) รอเล่นจบเกม
4. แก้ `CHANGE` เป็นสิ่งที่เปลี่ยน เช่น `"SPECIALTY 1.10"` แล้วรันช่องข้างล่างอีกครั้ง ตารางท้ายช่องจะสะสมผลทุกเกมไว้เทียบ

ช่องนี้เรียก `/v1/sandbox/reveal` ได้**หลังตลาดปิดเท่านั้น** และตลาดฝึกเริ่มเกมใหม่เองหลังปิด 5 นาที — รันภายใน 5 นาทีหลังเห็น `market_closed` (reveal มีเฉพาะตลาดฝึก วันแข่งไม่มี)

In [ ]:
CHANGE = "starter เดิม"            # ← เขียนว่าเกมนี้แก้อะไร ก่อนรันทุกครั้ง

r = api.get("/v1/sandbox/reveal")
if r.status_code != 200:
    print("ยังดูไม่ได้:", r.json()["error"]["message"])        # ตลาดยังไม่ปิด
else:
    rev = r.json()
    # ① แต่ละล็อต: ผลวัดตัวอย่าง (ที่เราเห็นตอนประเมิน) เทียบของจริงทั้งล็อต · TV จริง เทียบที่เราประเมิน
    for lot in rev["lots"]:
        s, t = lot["sample"], lot["true"]
        ye = lot["your_estimate"]
        est = ye.get("est_value_per_kg") if isinstance(ye, dict) else ye
        print(f"{lot['lot_id']}  ตัวอย่าง: บกพร่อง {s['defect_pct']}% ชื้น {s['moisture_pct']}% ขนาด {s['screen']}"
              f"  →  จริง: บกพร่อง {t['defect_pct']}% ชื้น {t['moisture_pct']}% ขนาด {t['screen']} · เกรดจริง {lot['true_grade']}")
        if est:
            err = (est - lot["true_value_per_kg"]) / lot["true_value_per_kg"]
            print(f"{'':<11}TV จริง {lot['true_value_per_kg']:.2f} · เราประเมิน {est:.2f} ({'สูงไป' if err > 0 else 'ต่ำไป'} {abs(err):.1%})")
        else:
            print(f"{'':<11}TV จริง {lot['true_value_per_kg']:.2f} · ไม่ได้ส่งใบประเมิน")
    # ② คะแนนของเกมนี้
    sc = rev["score"]
    mape = f"{sc['mape']:.1%}" if sc.get("mape") is not None else "-"
    print(f"\nPR {sc['pr']:.4f} · ประเมินคลาดเฉลี่ย {mape}")
    print(f"คะแนนขาย {sc['pr_score']:.1f}/80 + คะแนนประเมิน {sc['appraisal_score']:.1f}/20 = {sc['total']:.1f}")
    # ③ สะสมผลไว้เทียบหลายเกม · รันซ้ำในเกมเดิมแค่แก้ชื่อ ไม่นับเพิ่ม (หายเมื่อ restart kernel: จดลงกระดาษด้วย)
    trials = globals().get("trials", {})
    trials[tuple(l["true_value_per_kg"] for l in rev["lots"])] = (CHANGE, sc["pr"], mape)
    print("\nเทียบทุกเกมที่รันช่องนี้:")
    for i, (change, pr, err) in enumerate(trials.values(), 1):
        print(f"  เกม {i}: PR {pr:.4f} · ประเมินคลาด {err:>6} · {change}")

**อ่านผลอย่างไร**

- **แต่ละล็อต**: บรรทัดแรก = ผลวัดตัวอย่างที่เราเห็นตอนประเมิน เทียบกับค่าจริงของทั้งล็อต · บรรทัดที่สอง = TV จริง เทียบกับที่เราประเมิน สูงไปหรือต่ำไปกี่ % · ล็อตที่คลาดมากมักเป็นล็อตที่ตัวอย่างอยู่ใกล้ขั้นเกรด
- **PR** (Performance Ratio) = (รายได้ทั้งหมด + 0.80 × TV จริงของเมล็ดที่ขายไม่หมด) ÷ TV จริงของเมล็ดทั้งหมด
  - PR = 1.00 คือขายได้เท่ามูลค่าจริงพอดี · มากกว่า 1 = ขายได้แพงกว่ามูลค่า · ต่ำสุดราว 0.80 (ไม่ได้ขายเลย ได้แต่ค่าเศษ)
- **คะแนนขาย** (เต็ม 80) = 80 × (PR − 0.80) ÷ (PR ที่ดีที่สุดในตลาดเดียวกัน − 0.80) — ขึ้นกับว่าคู่แข่งในเกมนั้นเก่งแค่ไหนด้วย **เทียบเกมด้วย PR แทน**
- **คะแนนประเมิน** (เต็ม 20) ยิ่งประเมิน TV ใกล้ของจริงยิ่งได้มาก · ได้ 0 เมื่อคลาดเฉลี่ย 10% ขึ้นไป
- **ตารางเทียบ**: แต่ละเกมได้ล็อตและผู้ซื้อสุ่มใหม่ ผลจึงแกว่ง — เล่นอย่างน้อย 2–3 เกมต่อการแก้หนึ่งครั้งก่อนสรุปว่าดีขึ้นจริง

**ลองเอง / คุยกัน**

- ดูผลในข้อ 4.3: ประเภทผู้ซื้อไหนโดน `ABOVE_LIMIT` บ่อย ประเภทไหน `ACCEPTED` ง่ายเกินไป → ควรแก้ `MARKUP` ทางไหน
- ปิด LLM (`USE_LLM=0` ใน `.env`) แล้วเทียบ PR กับตอนเปิด — LLM ช่วยจริงไหม
- ถ้าได้ `SPEC_MISMATCH` ล็อตนั้นตัวอย่างห่างจากขอบสเปกเท่าไร

## ก่อนแข่งช่วงบ่าย

- บ่ายนี้ **ห้ามส่งคำสั่งด้วยมือ** — agent ต้องทำเองทั้งหมด หน้าเว็บดูได้อย่างเดียว
- ใช้ key ของวันแข่ง (คนละ key กับตลาดฝึก)
- ทดสอบ starter กับตลาดฝึกให้จบเกมได้ก่อน แล้วค่อยแก้ทีละจุด
- เรียก API ไม่เกิน 5 ครั้งต่อวินาที · เรียก LLM ไม่เกินที่ gateway กำหนด